# Beispiele: Regression

Gradient Descent, lineare Regression, Polynom-Regression, Ridge, Lasso und ElasticNet. Datensatz: `load_diabetes`, 442 Patientinnen und Patienten, 10 bereits skalierte Merkmale, Ziel ist der Krankheitsverlauf nach einem Jahr (eine Zahl).

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten sind in scikit-learn eingebaut oder werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/01_Regression.ipynb) (beim ersten Mal mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

**Mehr dazu in eigenen Notebooks:**
- [Supervised_Learning_1_Uebungen_own.ipynb](../Supervised_Learning_1_Uebungen_own.ipynb) ([in Colab](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/Supervised_Learning_1_Uebungen_own.ipynb)): dieselben Themen als Übungen mit Lücken und Prüfzellen

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## Daten

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X.head()

## Gradient Descent

So lernt ein Modell seine Gewichte: kleine Schritte gegen die Steigung. Hier für `f(x) = x²` (Steigung `2x`) mit drei Lernraten.

In [ ]:
def gradient_descent(lr, x0=-4.0, steps=50):
    x = x0
    for _ in range(steps):
        x = x - lr * 2 * x
    return x


for lr in [0.2, 1.0, 1.1]:
    print(lr, gradient_descent(lr))

# 0.2: läuft gegen 0 (Minimum) · 1.0: springt zwischen -4 und 4 · 1.1: divergiert

## Lineare Regression

**Frage:** Wie gut sagt eine Gerade (Ebene) den Krankheitsverlauf vorher? Trainiert wird auf 80 %, bewertet auf den 20 % Testdaten.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
model = LinearRegression().fit(X_train, y_train)
r2_test = r2_score(y_test, model.predict(X_test))
print(f"R² auf Testdaten: {r2_test:.3f}")
print(f"MAE auf Testdaten: {mean_absolute_error(y_test, model.predict(X_test)):.1f}")

# Die Koeffizienten zeigen, wie stark jedes Merkmal die Vorhersage verschiebt
pd.Series(model.coef_, index=X.columns).sort_values()

**Deutung:** R² um 0,45 heisst: Die Gerade erklärt knapp die Hälfte der Streuung. Die grössten Gewichte haben `s1` (negativ), `s5` und `bmi`. Die Merkmale sind in diesem Datensatz schon skaliert, darum lassen sich die Gewichte direkt vergleichen. Dass `s1` und `s2` gross und gegenläufig sind, ist ein Hinweis auf zwei stark korrelierte Merkmale (siehe Ridge und Lasso).

## Polynom-Regression und Overfitting

Polynome vom Grad 1 bis 10 auf verrauschten Sinus-Daten. Trainingsfehler und Testfehler pro Grad zeigen Under- und Overfitting.

In [ ]:
rng2 = np.random.default_rng(0)
x = np.sort(rng2.uniform(-1, 1, 30))
yy = np.sin(2 * np.pi * x) + rng2.normal(0, 0.3, 30)
xt, xe, yt, ye = train_test_split(x.reshape(-1, 1), yy, test_size=0.5, random_state=0)

train_mse, test_mse = {}, {}
for deg in range(1, 11):
    m = Pipeline([
        ("poly", PolynomialFeatures(deg, include_bias=False)),
        ("model", LinearRegression()),
    ])
    m.fit(xt, yt)
    train_mse[deg] = mean_squared_error(yt, m.predict(xt))
    test_mse[deg] = mean_squared_error(ye, m.predict(xe))

best_deg = min(test_mse, key=test_mse.get)
print(best_deg)

plt.plot(list(train_mse), list(train_mse.values()), "o-", label="Training")
plt.plot(list(test_mse), list(test_mse.values()), "o-", label="Test")
plt.yscale("log"); plt.xlabel("Grad"); plt.ylabel("MSE"); plt.legend(); plt.title("Under- und Overfitting")
plt.show()

**Deutung:** Der Trainingsfehler sinkt mit jedem Grad. Der Testfehler sinkt zuerst (weniger Bias) und steigt dann wieder (mehr Variance). Der beste Grad liegt dazwischen.

## Ridge und Lasso

**Lasso streicht, Ridge schrumpft:** Lasso setzt bei grösserem `alpha` immer mehr Koeffizienten auf genau 0. Ridge macht alle kleiner, aber keinen 0.

In [ ]:
Xs = StandardScaler().fit_transform(X_train)

n_zero = {}
for a in [0.01, 0.1, 1.0]:
    lasso = Lasso(alpha=a, max_iter=10_000).fit(Xs, y_train)
    n_zero[a] = int(np.sum(lasso.coef_ == 0))

ridge_sum = np.abs(Ridge(alpha=10).fit(Xs, y_train).coef_).sum()
linear_sum = np.abs(LinearRegression().fit(Xs, y_train).coef_).sum()
print(n_zero, ridge_sum, linear_sum)

In [ ]:
# Dieselbe Idee als Tabelle: Koeffizienten bei verschiedenen alpha
koef = pd.DataFrame({
    "Linear": LinearRegression().fit(Xs, y_train).coef_,
    "Ridge α=10": Ridge(alpha=10).fit(Xs, y_train).coef_,
    "Lasso α=1": Lasso(alpha=1).fit(Xs, y_train).coef_,
    "Lasso α=5": Lasso(alpha=5).fit(Xs, y_train).coef_,
}, index=X.columns)
koef.round(1)

**Deutung:** Ridge drückt vor allem das gegenläufige Paar `s1`/`s2` zusammen, das die lineare Regression gross gemacht hat. Lasso mit `alpha = 5` streicht vier Merkmale ganz.

## ElasticNet

Mischung aus Ridge und Lasso. `l1_ratio = 0` ist Ridge, `1` ist Lasso. Gesucht werden `alpha` und `l1_ratio` gemeinsam mit Cross-Validation.

In [ ]:
from sklearn.linear_model import ElasticNet

pipe_en = Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10_000))])
grid_en = {"model__alpha": [0.01, 0.1, 1], "model__l1_ratio": [0.1, 0.5, 0.9]}
search_en = GridSearchCV(pipe_en, grid_en, cv=5, scoring="r2").fit(X_train, y_train)

print("beste Kombination:", search_en.best_params_)
print(f"R² CV: {search_en.best_score_:.3f}")
coef_en = search_en.best_estimator_.named_steps["model"].coef_
print("Koeffizienten = 0:", int(np.sum(coef_en == 0)), "von", len(coef_en))